# 1. Data Ingestion and Connectors

In a real SOC, the first step is getting data INTO your SIEM. In Sentinel, this is done via **data connectors**. In our mini-SIEM, we have a REST API that ingests logs into named tables.

## Setup

```bash
cd security/sc-200/01-build-a-siem
docker compose up -d
uv sync
```

The log generator has already seeded data including attack patterns. Let's explore.

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

def pp(r):
    print(json.dumps(r.json(), indent=2))

# Check what's in our SIEM
print('=== SIEM Dashboard ===')
pp(httpx.get(f'{SIEM}/dashboard'))

## Understanding data tables

Our SIEM has 4 tables, matching real Sentinel tables:

| Our table | Real Sentinel table | What it contains |
|-----------|--------------------|-----------------|
| `SigninLogs` | `SigninLogs` | Entra ID sign-in events |
| `AzureFirewall` | `AzureDiagnostics` (filtered) | Firewall allow/deny decisions |
| `DeviceEvents` | `DeviceEvents` | Endpoint process executions, file operations |
| `EmailEvents` | `EmailEvents` | Email delivery, phishing detection |

Each data connector in Sentinel populates one or more tables. Let's query each one.

In [ ]:
# Query sign-in logs
print('=== Recent Sign-in Logs (last 5) ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'limit': 5,
})
for log in r.json()['results']:
    status = '✅' if log['ResultType'] == 'Success' else '❌'
    print(f'  {status} {log["UserPrincipalName"]:<25} {log["IPAddress"]:<16} {log["Location"]:<10} {log["AppDisplayName"]}')

In [ ]:
# Query firewall logs
print('=== Firewall Logs — Denied traffic ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'AzureFirewall',
    'filter': {'Action': 'Deny'},
    'limit': 10,
})
denied = r.json()['results']
if denied:
    for log in denied:
        print(f'  🚫 {log["SourceIP"]} → {log["DestinationIP"]}:{log["DestinationPort"]}')
else:
    print('  No denied traffic (all attacks went through — that\'s a problem!)')

# Show allowed outbound to suspicious IPs
print('\n=== Firewall: traffic to known bad IP ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'AzureFirewall',
    'filter': {'DestinationIP': '185.220.101.42'},
    'limit': 5,
})
for log in r.json()['results']:
    print(f'  ⚠️  {log["SourceIP"]} → {log["DestinationIP"]}:{log["DestinationPort"]} [{log["Action"]}]')

In [ ]:
# Aggregation query — like KQL's "summarize count() by UserPrincipalName"
print('=== Failed sign-ins by user (like KQL: summarize count() by UserPrincipalName) ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'filter': {'ResultType': 'Failure'},
    'aggregate_by': 'UserPrincipalName',
})
for row in r.json()['results']:
    bar = '█' * min(row['count'], 30)
    alert = ' ⚠️ BRUTE FORCE?' if row['count'] > 5 else ''
    print(f'  {row["group_key"]:<30} {row["count"]:>3} {bar}{alert}')

In [ ]:
# Ingest your own data — simulating a custom data connector
print('=== Custom data connector: ingest your own logs ===')

custom_logs = [
    {'table_name': 'CustomAppLogs', 'data': {'action': 'login', 'user': 'admin', 'ip': '10.0.1.50', 'status': 'success'}},
    {'table_name': 'CustomAppLogs', 'data': {'action': 'login', 'user': 'admin', 'ip': '185.220.101.42', 'status': 'failure'}},
    {'table_name': 'CustomAppLogs', 'data': {'action': 'export_data', 'user': 'admin', 'records': 50000, 'destination': 'external'}},
]

r = httpx.post(f'{SIEM}/ingest/batch', json={'entries': custom_logs})
print(f'Ingested {r.json()["count"]} custom log entries')

r = httpx.post(f'{SIEM}/query', json={'table_name': 'CustomAppLogs', 'limit': 10})
print(f'\nCustomAppLogs table now has {r.json()["count"]} entries:')
for log in r.json()['results']:
    print(f'  {log}')

## Real Sentinel data connectors

In the exam, you need to know WHICH connector to use:

| Data source | Connector type | Collection method |
|------------|----------------|-------------------|
| Entra ID sign-in logs | Built-in (1-click) | Direct API |
| Microsoft 365 | Built-in | Direct API |
| Azure Activity | Diagnostic settings | Azure Monitor pipeline |
| Windows Security Events | AMA (Azure Monitor Agent) | Data Collection Rule (DCR) |
| Linux Syslog | AMA + Syslog | Data Collection Rule |
| CEF (third-party firewalls) | AMA + CEF | Log forwarder → DCR |
| Custom application logs | Custom logs via API | REST API / DCR |

### Exam tip: AMA vs legacy agents

- **AMA (Azure Monitor Agent)** — the new standard. Uses DCRs. Supports multi-homing.
- **Log Analytics agent (MMA)** — deprecated. Still in exam questions about migration.
- **CEF/Syslog via AMA** — requires a Linux forwarder VM with the AMA extension.

**Next**: [Notebook 2 — Analytics Rules and Detection](02_analytics_rules.ipynb)